In [2]:
import os
import glob
import h5py
import cupy as cp
import numpy as np

DATA_DIR = "radioml2018"

# Automatically find the HDF5 file inside the directory
h5_files = glob.glob(os.path.join(DATA_DIR, "*.hdf5")) + glob.glob(
    os.path.join(DATA_DIR, "*.h5")
)
if not h5_files:
    raise FileNotFoundError(f"No .hdf5 or .h5 file found in directory: {DATA_DIR}")

FILE_PATH = h5_files[0]
print(f"Target file: {FILE_PATH}")


def test_index_envelope(target_onehot_idx, chunk_size=50000):
    """
    Streams small metadata slices to locate a +30 dB SNR sample,
    reads only that single frame into GPU memory, and evaluates envelope dynamics.
    """
    found_sample = None

    with h5py.File(FILE_PATH, "r") as f:
        total_samples = f["Y"].shape[0]

        for start in range(0, total_samples, chunk_size):
            end = min(start + chunk_size, total_samples)

            # Slice minimal 1D vectors (~50 KB host memory per slice)
            y_chunk = f["Y"][start:end, target_onehot_idx]
            z_chunk = f["Z"][start:end, 0]

            match = np.where((y_chunk == 1) & (z_chunk == 30))[0]
            if len(match) > 0:
                global_idx = start + match[0]
                # Pull only this single 1024x2 array into memory
                found_sample = f["X"][global_idx]
                break

    if found_sample is None:
        raise ValueError(
            f"No sample found for index {target_onehot_idx} at +30 dB SNR."
        )

    # Transfer single sample to GPU (1024, 2)
    iq_gpu = cp.asarray(found_sample, dtype=cp.float32)
    i = iq_gpu[:, 0]
    q = iq_gpu[:, 1]

    # Calculate envelope: sqrt(I^2 + Q^2)
    env = cp.sqrt(i**2 + q**2)

    # Check for OOK zero-energy intervals
    max_env = cp.max(env)
    zero_fraction = float(cp.mean(env < (0.15 * max_env)))
    mean_val = float(cp.mean(env))
    std_val = float(cp.std(env))
    cv = std_val / (mean_val + 1e-8)

    return {"mean": mean_val, "std": std_val, "cv": cv, "zero_fraction": zero_fraction}


print("\n--- Testing Index 0 ---")
res_0 = test_index_envelope(target_onehot_idx=0)
print(f"Zero-energy ratio : {res_0['zero_fraction'] * 100:.2f}%")
print(f"Variation (std/mean): {res_0['cv']:.3f}")

if res_0["zero_fraction"] > 0.30 or res_0["cv"] > 0.6:
    print("Verdict for Index 0 -> OOK (classes-fixed is physical storage)")
else:
    print("Verdict for Index 0 -> 32PSK (classes is physical storage)")

print("\n--- Testing Index 22 ---")
res_22 = test_index_envelope(target_onehot_idx=22)
print(f"Zero-energy ratio : {res_22['zero_fraction'] * 100:.2f}%")
print(f"Variation (std/mean): {res_22['cv']:.3f}")

if res_22["zero_fraction"] > 0.30 or res_22["cv"] > 0.6:
    print("Verdict for Index 22 -> OOK (classes is physical storage)")
else:
    print(
        "Verdict for Index 22 -> GMSK / Continuous (classes-fixed is physical storage)"
    )

Target file: radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5

--- Testing Index 0 ---
Zero-energy ratio : 26.46%
Variation (std/mean): 0.674
Verdict for Index 0 -> OOK (classes-fixed is physical storage)

--- Testing Index 22 ---
Zero-energy ratio : 0.00%
Variation (std/mean): 0.071
Verdict for Index 22 -> GMSK / Continuous (classes-fixed is physical storage)
